In [ ]:
import scipy.io
import numpy as np
import mne
import matplotlib.pyplot as plt
import addcopyfighandler
%matplotlib qt

# mat file 불러오기
mat = scipy.io.loadmat(r'\\HCN-NAS\ywYang\s14000.mat')
print(mat.keys())
fields = list(mat['eeg'][0].dtype.fields.keys())
print(fields)

dict_keys(['__header__', '__version__', '__globals__', 'eeg'])
['raw_left', 'raw_right', 'n_trials', 'frame', 'srate', 'event', 'senloc', 'psenloc', 'subject', 'comment', 'emg_left', 'emg_right', 'rest', 'noise']


In [2]:
#eeg 채널 이름 정의
eeg_ch_names = [
    "Fp1", "AF7", "AF3", "F1", "F3", "F5", "F7", "FT7", "FC5","FC3", "FC1", 
    "C1", "C3", "C5", "T7", "TP7", "CP5", "CP3", "CP1", "P1", "P3", "P5", "P7", 
    "P9", "PO7", "PO3", "O1", "Iz", "Oz", "POz", "Pz", "CPz", "Fpz", "Fp2", 
    "AF8", "AF4", "AFz", "Fz", "F2", "F4", "F6", "F8", "FT8", "FC6", "FC4",
    "FC2", "FCz", "Cz", "C2", "C4", "C6", "T8", "TP8", "CP6", "CP4", "CP2", 
    "P2", "P4", "P6", "P8", "P10", "PO8", "PO4", "O2",
]
#emg 채널 이름 정의
emg_ch_names = ["EMG1", "EMG2", "EMG3", "EMG4"]

#stim이라는 채널과 eeg, emg 채널들 결합
ch_names = eeg_ch_names + emg_ch_names + ["stim"]

#채널 유형 정의 -> eeg 채널 64개, emg 채널 4개, stim 채널 1개
ch_types = ["eeg"] * 64 + ["emg"] * 4 + ["stim"] * 1

sampling_freq = 512

In [ ]:
#info 생성
info = mne.create_info(ch_names=ch_names, ch_types=ch_types, sfreq=sampling_freq)
info["description"] = "Motor Imagery dataset"
info["bads"] = []

left = np.vstack((mat['eeg'][0]['raw_left'][0], mat['eeg'][0]['emg_left'][0], mat['eeg'][0]['event'][0]))
right = np.vstack((mat['eeg'][0]['raw_right'][0], mat['eeg'][0]['emg_right'][0], mat['eeg'][0]['event'][0]))

# raw 객체 생성
lr = mne.io.RawArray(left, info=info)
rr = mne.io.RawArray(right, info=info)
raw = {
    "left": lr,
    "right": rr
}

LEFT = 0
RIGHT = 1

Creating RawArray with float64 data, n_channels=69, n_times=358400
    Range : 0 ... 358399 =      0.000 ...   699.998 secs
Ready.
Creating RawArray with float64 data, n_channels=69, n_times=358400
    Range : 0 ... 358399 =      0.000 ...   699.998 secs
Ready.


NameError: name 'fnames' is not defined

In [4]:
montage = mne.channels.make_standard_montage("standard_1020")

raw["left"].set_montage(montage, on_missing = "ignore")
raw["right"].set_montage(montage, on_missing = "ignore")

<RawArray | 69 x 358400 (700.0 s), ~188.7 MiB, data loaded>

In [5]:
raw["left"].set_eeg_reference("average", projection = False)
raw["right"].set_eeg_reference("average", projection = False)

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.


<RawArray | 69 x 358400 (700.0 s), ~188.7 MiB, data loaded>

In [16]:
raw["left"].filter(l_freq= 8, h_freq = 30)
raw["right"].filter(l_freq = 8, h_freq = 30)

raw["left"].notch_filter(freqs = 60)
raw["right"].notch_filter(freqs = 60) 

raw["left"].plot()
raw["right"].plot()

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 845 samples (1.650 s)

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 30

<mne_qt_browser._pg_figure.MNEQtBrowser(0x22286e02f90) at 0x0000022281125A00>

In [ ]:
le = mne.find_events(raw["left"])
re = mne.find_events(raw["right"])
events = [le, re]

Finding events on: stim
100 events found on stim channel stim
Event IDs: [1]
Finding events on: stim
100 events found on stim channel stim
Event IDs: [1]


In [9]:
le = mne.Epochs(raw["left"],
                events[0], 
                tmin = -2, tmax = 5, 
                picks = ("C3", "C4"),
                baseline = (-0.5, 0),
                preload=True,)
re = mne.Epochs(raw["right"],
                events[1],
                tmin = -2, tmax = 5,
                picks = ["C3", "C4"],
                baseline = (-0.5, 0),
                preload = True,)
epochs = [le, re]

reject_criteria = dict(eeg=100e-6)

Not setting metadata
100 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 100 events and 3585 original time points ...
1 bad epochs dropped
Not setting metadata
100 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 100 events and 3585 original time points ...
1 bad epochs dropped


In [10]:
epochs[LEFT].plot(
    n_epochs = 2,
    n_channels = 2,
    title = "Left MI epochs",
    block = False,
)

Using qt as 2D backend.


<mne_qt_browser._pg_figure.MNEQtBrowser(0x222c2bcb920) at 0x0000022281D97180>